In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)

In [ ]:
from ultralytics import YOLO

# Load a pre-trained small YOLO model to tap into transfer learning feature weights
model = YOLO("yolov8s.pt")

# Train the network using your automated synthetic data configurations
results = model.train(
    data="./dataset.yaml",
    epochs=80,         
    imgsz=640,         # Matches 640x640 Blender viewport settings
    batch=16,          # Safe batch size for standard laptop GPU memory boundaries
    device="0" ,
    
)

In [ ]:
import os
import glob
from ultralytics import YOLO

runs_dir = "runs/detect"
test_image = "real_test_2.jpeg"  # pick one consistent test image
output_dir = "checkpoint_comparison"
os.makedirs(output_dir, exist_ok=True)

# Find every train* folder that has a best.pt
weight_paths = sorted(glob.glob(os.path.join(runs_dir, "train*", "weights", "best.pt")))

print(f"Found {len(weight_paths)} checkpoints:")
for wp in weight_paths:
    print(" -", wp)

for wp in weight_paths:
    run_name = wp.split(os.sep)[-3]  # e.g. "train-4"
    try:
        model = YOLO(wp)
        results = model.predict(source=test_image, conf=0.25, iou=0.5, verbose=False)
        r = results[0]
        annotated = r.plot()

        out_path = os.path.join(output_dir, f"{run_name}_pred.jpg")
        import cv2
        cv2.imwrite(out_path, annotated)

        # Also print class names + params to identify the model
        print(f"\n--- {run_name} ---")
        print(f"  classes: {model.names}")
        print(f"  detections: {len(r.boxes)}")
        for box in r.boxes:
            print(f"    {model.names[int(box.cls)]} conf={float(box.conf):.2f}")
    except Exception as e:
        print(f"  Failed on {run_name}: {e}")

In [ ]:
import os
import glob
import cv2
from ultralytics import YOLO

runs_dir = "runs/detect"
test_images = ["real_test_1.jpeg", "real_test_2.jpeg", "real_test_3.jpeg", "real_test_4.jpeg"]
output_dir = "checkpoint_comparison"
os.makedirs(output_dir, exist_ok=True)

weight_paths = sorted(glob.glob(os.path.join(runs_dir, "train*", "weights", "best.pt")))

summary = {}

for wp in weight_paths:
    run_name = wp.split(os.sep)[-3]
    model = YOLO(wp)
    summary[run_name] = {}
    for img in test_images:
        if not os.path.exists(img):
            continue
        results = model.predict(source=img, conf=0.25, iou=0.5, verbose=False)
        r = results[0]
        dets = [(model.names[int(b.cls)], round(float(b.conf), 2)) for b in r.boxes]
        summary[run_name][img] = dets

        annotated = r.plot()
        cv2.imwrite(os.path.join(output_dir, f"{run_name}_{img.replace('.jpeg','')}.jpg"), annotated)

print("\n=== SUMMARY: checkpoint x test image ===")
for run_name, imgs in summary.items():
    print(f"\n{run_name}:")
    for img, dets in imgs.items():
        print(f"  {img}: {dets}")

In [ ]:
import os
import glob
import cv2
from ultralytics import YOLO

runs_dir = "runs/detect"
test_images = ["real_test_1.jpeg", "real_test_2.jpeg", "real_test_3.jpeg", "real_test_4.jpeg"]
output_dir = "checkpoint_comparison"
os.makedirs(output_dir, exist_ok=True)

weight_paths = sorted(glob.glob(os.path.join(runs_dir, "train*", "weights", "best.pt")))

summary = {}

for wp in weight_paths:
    run_name = wp.split(os.sep)[-3]
    model = YOLO(wp)
    summary[run_name] = {}
    for img in test_images:
        if not os.path.exists(img):
            continue
        results = model.predict(source=img, conf=0.50, iou=0.5, verbose=False)
        r = results[0]
        dets = [(model.names[int(b.cls)], round(float(b.conf), 2)) for b in r.boxes]
        summary[run_name][img] = dets

        annotated = r.plot()
        cv2.imwrite(os.path.join(output_dir, f"{run_name}_{img.replace('.jpeg','')}.jpg"), annotated)

print("\n=== SUMMARY: checkpoint x test image ===")
for run_name, imgs in summary.items():
    print(f"\n{run_name}:")
    for img, dets in imgs.items():
        print(f"  {img}: {dets}")

In [ ]:
from ultralytics import YOLO
model = YOLO("runs/detect/train-8/weights/best.pt")
metrics = model.val(data="real_eval.yaml", split="val")
print("Real-world mAP50:", metrics.box.map50)

In [ ]:
from ultralytics import YOLO
model = YOLO("runs/detect/train-6/weights/best.pt")
metrics = model.val(data="real_eval.yaml", split="val")
print("Real-world mAP50:", metrics.box.map50)